In [15]:
import json
import os
import pandas as pd

train_path = "D:\.Projects\\Topic202507Codes\\data\\redial_dataset\\train_data.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
train_data = [json.loads(line) for line in lines]

test_path = "D:\.Projects\\Topic202507Codes\\data\\redial_dataset\\test_data.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
test_data = [json.loads(line) for line in lines]

movie_id2title = {}
movie_title2id = {}
movie_path = "D:\.Projects\\Topic202507Codes\\data\\redial_dataset\\movies_with_mentions.csv"
movie_mentions = pd.read_csv(movie_path)
for _, row in movie_mentions.iterrows():
    id = row["movieId"]
    title = row["movieName"]
    movie_id2title[id] = title
    movie_title2id[title] = id

In [16]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [23]:
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from tqdm import tqdm
import re

# def clean_text(text):\


def process(data, movie_patter=r"@\d+"):
    processed_data = []
    tokenizer = RegexpTokenizer(r"@\d+|[\w\']+|[^\w\s]")
    stop_words = set(stopwords.words("english"))
    for conv in tqdm(data):
        conv_dict = {}
        conv_seeker = conv["initiatorWorkerId"]
        conv_dict["conv_id"] = conv["conversationId"]
        conv_dict["dialog"] = []
        for utt_id, utt in enumerate(conv["messages"]):
            utt_dict = {}
            utt_dict["utt_id"] = utt_id
            utt_dict["role"] = "Seeker" if utt["senderWorkerId"] == conv_seeker else "Recommender"

            movies = re.findall(movie_patter, utt["text"])
            movies_ids = [int(mv[1:]) for mv in movies if int(mv[1:]) in movie_id2title]
            utt_dict["movies"] = movies_ids

            tokens = tokenizer.tokenize(utt["text"])
            filtered_tokens = [word.lower() for word in tokens if word.lower() not in stop_words]
            utt_dict["text"] = filtered_tokens

            conv_dict["dialog"].append(utt_dict)
        processed_data.append(conv_dict)
    return processed_data

In [24]:
processed_train = process(train_data)
processed_test = process(test_data)

100%|██████████| 10006/10006 [00:02<00:00, 4047.13it/s]


In [25]:
print(json.dumps(processed_train[0], indent=2))

{
  "conv_id": "391",
  "dialog": [
    {
      "utt_id": 0,
      "role": "Seeker",
      "movies": [],
      "text": [
        "hi",
        ",",
        "?",
        "looking",
        "movie",
        "recommendations"
      ]
    },
    {
      "utt_id": 1,
      "role": "Recommender",
      "movies": [],
      "text": [
        "okay",
        ".",
        "kind",
        "movies",
        "like",
        "?"
      ]
    },
    {
      "utt_id": 2,
      "role": "Seeker",
      "movies": [
        84779,
        191602
      ],
      "text": [
        "like",
        "animations",
        "like",
        "@84779",
        "@191602"
      ]
    },
    {
      "utt_id": 3,
      "role": "Seeker",
      "movies": [
        122159
      ],
      "text": [
        "also",
        "enjoy",
        "@122159"
      ]
    },
    {
      "utt_id": 4,
      "role": "Seeker",
      "movies": [],
      "text": [
        "anything",
        "artistic"
      ]
    },
    {
      "utt_id": 5,
  

In [29]:
output_dir = os.path.join(os.getcwd(), "redial")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "train_data.json"), "w", encoding="utf-8") as f:
    json.dump(processed_train, f, indent=2)
with open(os.path.join(output_dir, "test_data.json"), "w", encoding="utf-8") as f:
    json.dump(processed_test, f, indent=2)

In [30]:
token2id = {"__pad__": 0, "__start__": 1, "__end__": 2, "__unk__": 3}
for conv in processed_train + processed_test:
    for utt in conv["dialog"]:
        for token in utt["text"]:
            if token not in token2id:
                token2id[token] = len(token2id)
with open(os.path.join(output_dir, "token2id.json"), "w", encoding="utf-8") as f:
    json.dump(token2id, f, indent=2)

# Step 1: 构建推荐任务数据集

从对话中提取 `(context → recommended_movie)` 训练对。
- 上下文：当前轮之前的所有对话历史
- 标签：Recommender 在本轮提到的电影

In [ ]:
def build_recommendation_samples(processed_data, token2id, max_context_len=256):
    """
    从处理过的对话数据构建推荐任务样本
    
    Returns:
        rec_samples: List[Dict] with keys:
            - context_tokens: List[int] - 上下文的 token IDs
            - item: int - 推荐的电影ID（在 item2id 中的索引）
            - conv_id: int - 对话ID
            - turn_id: int - 对话轮次
    """
    rec_samples = []
    
    for conv in tqdm(processed_data, desc="Building recommendation samples"):
        context = []  # 累积对话上下文
        
        for turn in conv['dialog']:
            # 只在 Recommender 提及电影时创建训练样本
            if turn['role'] == 'Recommender' and len(turn['movies']) > 0:
                # 为每个提及的电影创建一个样本
                for movie_id in turn['movies']:
                    sample = {
                        'context_tokens': context[-max_context_len:],  # 截断到最大长度
                        'item': movie_id,  # 原始电影ID（稍后会映射到索引）
                        'conv_id': conv['conv_id'],
                        'turn_id': turn['utt_id']
                    }
                    rec_samples.append(sample)
            
            # 更新上下文：添加当前轮的文本
            # 格式: [role_token] + tokens
            role_token = '[seeker]' if turn['role'] == 'Seeker' else '[recommender]'
            turn_tokens = [role_token] + turn['text']
            
            # 转换为 token IDs
            turn_ids = [token2id.get(t, token2id['__unk__']) for t in turn_tokens]
            context.extend(turn_ids)
    
    return rec_samples

# 构建训练集和测试集
train_rec_samples = build_recommendation_samples(processed_train, token2id)
test_rec_samples = build_recommendation_samples(processed_test, token2id)

print(f"推荐训练样本数: {len(train_rec_samples)}")
print(f"推荐测试样本数: {len(test_rec_samples)}")
print(f"\n示例样本:")
print(f"  上下文长度: {len(train_rec_samples[0]['context_tokens'])}")
print(f"  推荐电影ID: {train_rec_samples[0]['item']}")
print(f"  对话ID: {train_rec_samples[0]['conv_id']}")

# Step 2: 构建对话任务数据集

从对话中提取 `(context → response)` 训练对。
- 上下文：当前轮之前的对话
- 标签：当前轮的完整回复

In [ ]:
def build_conversation_samples(processed_data, token2id, max_context_len=256, max_response_len=50):
    """
    从处理过的对话数据构建对话生成任务样本
    
    Returns:
        conv_samples: List[Dict] with keys:
            - context_tokens: List[int] - 上下文的 token IDs
            - response: List[int] - 回复的 token IDs
            - conv_id: int - 对话ID
            - turn_id: int - 对话轮次
            - role: str - 发言角色
    """
    conv_samples = []
    
    for conv in tqdm(processed_data, desc="Building conversation samples"):
        context = []  # 累积对话上下文
        
        for turn in conv['dialog']:
            # 跳过第一轮（没有上下文）
            if turn['utt_id'] > 0:
                # 准备回复 tokens
                role_token = '[seeker]' if turn['role'] == 'Seeker' else '[recommender]'
                response_tokens = [role_token] + turn['text']
                response_ids = [token2id.get(t, token2id['__unk__']) for t in response_tokens]
                
                # 截断回复长度
                response_ids = response_ids[:max_response_len]
                
                sample = {
                    'context_tokens': context[-max_context_len:],
                    'response': response_ids,
                    'conv_id': conv['conv_id'],
                    'turn_id': turn['utt_id'],
                    'role': turn['role']
                }
                conv_samples.append(sample)
            
            # 更新上下文
            role_token = '[seeker]' if turn['role'] == 'Seeker' else '[recommender]'
            turn_tokens = [role_token] + turn['text']
            turn_ids = [token2id.get(t, token2id['__unk__']) for t in turn_tokens]
            context.extend(turn_ids)
    
    return conv_samples

# 构建训练集和测试集
train_conv_samples = build_conversation_samples(processed_train, token2id)
test_conv_samples = build_conversation_samples(processed_test, token2id)

print(f"对话训练样本数: {len(train_conv_samples)}")
print(f"对话测试样本数: {len(test_conv_samples)}")
print(f"\n示例样本:")
print(f"  上下文长度: {len(train_conv_samples[0]['context_tokens'])}")
print(f"  回复长度: {len(train_conv_samples[0]['response'])}")
print(f"  角色: {train_conv_samples[0]['role']}")

# Step 3: 构建 side_data

创建模型需要的辅助数据结构。

In [ ]:
def build_side_data(movie_id2title, train_samples, test_samples):
    """
    构建 side_data 字典
    
    Returns:
        side_data: Dict with keys:
            - n_entity: int - 电影总数
            - item2id: Dict[int, int] - 原始电影ID → 模型索引
            - id2item: List[int] - 模型索引 → 原始电影ID
            - item_meta: Dict - 电影元数据
    """
    # 收集所有出现的电影ID
    all_movie_ids = set()
    for samples in [train_samples, test_samples]:
        for sample in samples:
            all_movie_ids.add(sample['item'])
    
    # 创建 ID 映射
    item2id = {}
    id2item = []
    for idx, movie_id in enumerate(sorted(all_movie_ids)):
        item2id[movie_id] = idx
        id2item.append(movie_id)
    
    # 创建元数据
    item_meta = {}
    for idx, movie_id in enumerate(id2item):
        item_meta[idx] = {
            'original_id': movie_id,
            'title': movie_id2title.get(movie_id, f"Unknown_{movie_id}"),
            'mentions_count': 0  # 可以统计出现次数
        }
    
    # 统计电影出现次数
    for samples in [train_samples, test_samples]:
        for sample in samples:
            model_idx = item2id[sample['item']]
            item_meta[model_idx]['mentions_count'] += 1
    
    side_data = {
        'n_entity': len(id2item),
        'item2id': item2id,
        'id2item': id2item,
        'item_meta': item_meta
    }
    
    return side_data

# 构建 side_data
side_data = build_side_data(movie_id2title, train_rec_samples, test_rec_samples)

print(f"电影总数 (n_entity): {side_data['n_entity']}")
print(f"示例电影元数据:")
print(json.dumps(side_data['item_meta'][0], indent=2))

# 将推荐样本中的电影ID映射到模型索引
for sample in train_rec_samples + test_rec_samples:
    sample['item'] = side_data['item2id'][sample['item']]

print(f"\n✅ 已将电影ID映射到模型索引 [0, {side_data['n_entity']-1}]")

# Step 4: 添加角色 token 到词表

需要添加 `[seeker]` 和 `[recommender]` 作为特殊 token。

In [ ]:
# 添加角色 token
if '[seeker]' not in token2id:
    token2id['[seeker]'] = len(token2id)
if '[recommender]' not in token2id:
    token2id['[recommender]'] = len(token2id)

# 创建反向映射
id2token = {v: k for k, v in token2id.items()}

print(f"词表大小: {len(token2id)}")
print(f"特殊 tokens:")
print(f"  __pad__: {token2id['__pad__']}")
print(f"  __start__: {token2id['__start__']}")
print(f"  __end__: {token2id['__end__']}")
print(f"  __unk__: {token2id['__unk__']}")
print(f"  [seeker]: {token2id['[seeker]']}")
print(f"  [recommender]: {token2id['[recommender]']}")

# Step 5: 保存处理后的数据

将所有数据保存为 pickle 或 JSON 格式，方便训练时加载。

In [ ]:
import pickle

# 创建保存目录
processed_dir = os.path.join(os.getcwd(), "redial_processed")
os.makedirs(processed_dir, exist_ok=True)

# 保存推荐数据
with open(os.path.join(processed_dir, "train_rec.pkl"), "wb") as f:
    pickle.dump(train_rec_samples, f)
with open(os.path.join(processed_dir, "test_rec.pkl"), "wb") as f:
    pickle.dump(test_rec_samples, f)

# 保存对话数据
with open(os.path.join(processed_dir, "train_conv.pkl"), "wb") as f:
    pickle.dump(train_conv_samples, f)
with open(os.path.join(processed_dir, "test_conv.pkl"), "wb") as f:
    pickle.dump(test_conv_samples, f)

# 保存词表和 side_data
with open(os.path.join(processed_dir, "vocab.pkl"), "wb") as f:
    pickle.dump({
        'token2id': token2id,
        'id2token': id2token
    }, f)

with open(os.path.join(processed_dir, "side_data.pkl"), "wb") as f:
    pickle.dump(side_data, f)

# 也保存一份 JSON 格式方便查看
with open(os.path.join(processed_dir, "side_data.json"), "w", encoding="utf-8") as f:
    # 只保存可序列化的部分
    json_side_data = {
        'n_entity': side_data['n_entity'],
        'id2item': side_data['id2item'],
        'item_meta': {k: v for k, v in list(side_data['item_meta'].items())[:10]}  # 只保存前10个示例
    }
    json.dump(json_side_data, f, indent=2)

print("✅ 数据已保存到:", processed_dir)
print(f"\n文件列表:")
print(f"  - train_rec.pkl: {len(train_rec_samples)} 推荐训练样本")
print(f"  - test_rec.pkl: {len(test_rec_samples)} 推荐测试样本")
print(f"  - train_conv.pkl: {len(train_conv_samples)} 对话训练样本")
print(f"  - test_conv.pkl: {len(test_conv_samples)} 对话测试样本")
print(f"  - vocab.pkl: 词表 (大小: {len(token2id)})")
print(f"  - side_data.pkl: 辅助数据 ({side_data['n_entity']} 电影)")

# Step 6: 准备超图数据（可选但推荐）

如果你有预计算的图像/视频/音频 embeddings，可以构建超图数据结构。

In [ ]:
import torch
from torch_geometric.data import Data
import numpy as np

def load_hypergraph_features(movie_id, emb_dir="D:\.Projects\\Topic202507Codes\\data\\redial_dataset"):
    """
    加载单个电影的多模态 embedding
    
    Args:
        movie_id: 电影ID
        emb_dir: embedding 文件目录
    
    Returns:
        Dict[str, torch.Tensor] - 各模态的特征
    """
    features = {}
    
    # 尝试加载图像 embedding
    img_path = os.path.join(emb_dir, "img_emb", f"{movie_id}.pt")
    if os.path.exists(img_path):
        features['image'] = torch.load(img_path)
    
    # 尝试加载视频 embedding
    vdo_path = os.path.join(emb_dir, "vdo_emb", f"{movie_id}.pt")
    if os.path.exists(vdo_path):
        features['video'] = torch.load(vdo_path)
    
    # 尝试加载音频 embedding
    audio_path = os.path.join(emb_dir, "audio_emb", f"{movie_id}.pt")
    if os.path.exists(audio_path):
        features['audio'] = torch.load(audio_path)
    
    return features

def build_hypergraph_data(movie_id, features, method='knn', k=10):
    """
    从多模态特征构建超图 Data 对象
    
    Args:
        movie_id: 电影ID
        features: Dict[modality, tensor] - 各模态特征
        method: 超图构建方法 ('knn', 'threshold', 'complete')
        k: KNN 的 k 值
    
    Returns:
        Data or Dict[str, Data] - 超图数据
    """
    if len(features) == 0:
        return None
    
    # 简单方法：为每个模态创建一个节点，并添加自连接边
    # 更复杂的方法可以基于相似度构建超边
    
    if len(features) == 1:
        # 单模态：创建简单图
        modality, feat = list(features.items())[0]
        
        # 假设 feat 是 [num_patches, dim] 或 [dim]
        if feat.dim() == 1:
            feat = feat.unsqueeze(0)  # [1, dim]
        
        num_nodes = feat.size(0)
        
        # 创建全连接边（超图中所有节点互连）
        edge_index = torch.combinations(torch.arange(num_nodes), r=2).t()
        if edge_index.size(1) == 0:  # 只有一个节点
            edge_index = torch.tensor([[0], [0]], dtype=torch.long)
        
        data = Data(
            x=feat,
            edge_index=edge_index,
            num_nodes=num_nodes
        )
        return data
    
    else:
        # 多模态：为每个模态创建一个 Data 对象
        multi_modal_data = {}
        
        for modality, feat in features.items():
            if feat.dim() == 1:
                feat = feat.unsqueeze(0)
            
            num_nodes = feat.size(0)
            edge_index = torch.combinations(torch.arange(num_nodes), r=2).t()
            if edge_index.size(1) == 0:
                edge_index = torch.tensor([[0], [0]], dtype=torch.long)
            
            multi_modal_data[modality] = Data(
                x=feat,
                edge_index=edge_index,
                num_nodes=num_nodes
            )
        
        return multi_modal_data

# 示例：为一个电影构建超图
sample_movie_id = side_data['id2item'][0]
print(f"正在为电影 {sample_movie_id} ({movie_id2title.get(sample_movie_id, 'Unknown')}) 构建超图...")

features = load_hypergraph_features(sample_movie_id)
if features:
    print(f"找到的模态: {list(features.keys())}")
    for modality, feat in features.items():
        print(f"  {modality}: shape = {feat.shape}")
    
    hypergraph = build_hypergraph_data(sample_movie_id, features)
    
    if isinstance(hypergraph, Data):
        print(f"\n单模态超图:")
        print(f"  节点数: {hypergraph.num_nodes}")
        print(f"  边数: {hypergraph.edge_index.size(1)}")
        print(f"  特征维度: {hypergraph.x.size(1)}")
    else:
        print(f"\n多模态超图:")
        for modality, data in hypergraph.items():
            print(f"  {modality}: {data.num_nodes} 节点, {data.edge_index.size(1)} 边")
else:
    print("❌ 未找到该电影的 embedding 文件")
    print("提示: 你需要先运行 img.py, vdo.py, ado.py 等脚本提取特征")

# 📋 数据准备总结

## ✅ 已完成的工作

1. **推荐任务数据** (`train_rec.pkl`, `test_rec.pkl`)
   - 格式: `{'context_tokens': List[int], 'item': int, 'conv_id': int, 'turn_id': int}`
   - 数量: 根据 Recommender 提及电影的次数决定

2. **对话任务数据** (`train_conv.pkl`, `test_conv.pkl`)
   - 格式: `{'context_tokens': List[int], 'response': List[int], 'conv_id': int, 'turn_id': int, 'role': str}`
   - 数量: 每轮对话（除第一轮）一个样本

3. **词表** (`vocab.pkl`)
   - `token2id`: 词 → ID 映射
   - `id2token`: ID → 词 映射
   - 包含特殊 token: `__pad__`, `__start__`, `__end__`, `__unk__`, `[seeker]`, `[recommender]`

4. **辅助数据** (`side_data.pkl`)
   - `n_entity`: 电影总数
   - `item2id`: 原始电影ID → 模型索引
   - `id2item`: 模型索引 → 原始电影ID
   - `item_meta`: 电影元数据

5. **超图数据准备函数** (可选)
   - `load_hypergraph_features()`: 加载多模态 embeddings
   - `build_hypergraph_data()`: 构建 torch_geometric.Data

---

## 🔧 下一步：创建 DataLoader

你需要创建一个自定义 DataLoader 类，实现 `get_rec_data()` 和 `get_conv_data()` 方法。

### 推荐任务 DataLoader 需要返回的 batch:
```python
{
    'context_tokens': torch.LongTensor([batch_size, max_len]),
    'attention_mask': torch.FloatTensor([batch_size, max_len]),
    'item': torch.LongTensor([batch_size]),
    'graph_data': List[Data]  # 可选，每个样本对应的超图
}
```

### 对话任务 DataLoader 需要返回的 batch:
```python
{
    'context_tokens': torch.LongTensor([batch_size, context_len]),
    'response': torch.LongTensor([batch_size, response_len]),
    'attention_mask': torch.FloatTensor([batch_size, total_len]),
    'graph_data': List[Data]  # 可选
}
```

---

## 💡 关键要点

1. **Padding**: 在 collate_fn 中将不同长度的序列 pad 到相同长度
2. **超图数据**: 在训练时动态加载（避免内存占用过大）
3. **数据增强**: 可以考虑随机截断上下文、同义词替换等
4. **验证集**: 从训练集中划分 10-20% 作为验证集

---

## 📂 文件结构

```
redial_processed/
├── train_rec.pkl          # 推荐训练数据
├── test_rec.pkl           # 推荐测试数据
├── train_conv.pkl         # 对话训练数据
├── test_conv.pkl          # 对话测试数据
├── vocab.pkl              # 词表
├── side_data.pkl          # 辅助数据
└── side_data.json         # 辅助数据（JSON格式，方便查看）
```

# ⚠️ 重要更新：保存原始文本而非 Token IDs

**原因**：HypergraphLlava 使用 LLaVA 的 tokenizer（32K+ 词表），和我们的自建词表不兼容。

**解决方案**：重新处理数据，保存原始文本字符串。

In [ ]:
def build_recommendation_samples_v2(processed_data, max_context_len=256):
    """
    构建推荐样本 - 保存原始文本而非 token IDs
    
    Returns:
        rec_samples: List[Dict] with keys:
            - context_text: str - 对话上下文（原始文本）
            - item: int - 推荐的电影ID
            - conv_id: int
            - turn_id: int
    """
    rec_samples = []
    
    for conv in tqdm(processed_data, desc="Building recommendation samples (v2)"):
        context_tokens = []  # 累积对话上下文（token 列表）
        
        for turn in conv['dialog']:
            # 只在 Recommender 提及电影时创建训练样本
            if turn['role'] == 'Recommender' and len(turn['movies']) > 0:
                # 将 context tokens 转换为文本
                # 格式: "[Seeker] ... [Recommender] ..."
                context_text = ' '.join(context_tokens[-max_context_len:])
                
                for movie_id in turn['movies']:
                    sample = {
                        'context_text': context_text,
                        'item': movie_id,
                        'conv_id': conv['conv_id'],
                        'turn_id': turn['utt_id']
                    }
                    rec_samples.append(sample)
            
            # 更新上下文
            role_token = '[Seeker]' if turn['role'] == 'Seeker' else '[Recommender]'
            turn_text = [role_token] + turn['text']
            context_tokens.extend(turn_text)
    
    return rec_samples


def build_conversation_samples_v2(processed_data, max_context_len=256):
    """
    构建对话样本 - 保存原始文本
    
    Returns:
        conv_samples: List[Dict] with keys:
            - context_text: str
            - response_text: str
            - conv_id: int
            - turn_id: int
            - role: str
    """
    conv_samples = []
    
    for conv in tqdm(processed_data, desc="Building conversation samples (v2)"):
        context_tokens = []
        
        for turn in conv['dialog']:
            if turn['utt_id'] > 0:
                # Context
                context_text = ' '.join(context_tokens[-max_context_len:])
                
                # Response
                role_token = '[Seeker]' if turn['role'] == 'Seeker' else '[Recommender]'
                response_tokens = [role_token] + turn['text']
                response_text = ' '.join(response_tokens)
                
                sample = {
                    'context_text': context_text,
                    'response_text': response_text,
                    'conv_id': conv['conv_id'],
                    'turn_id': turn['utt_id'],
                    'role': turn['role']
                }
                conv_samples.append(sample)
            
            # Update context
            role_token = '[Seeker]' if turn['role'] == 'Seeker' else '[Recommender]'
            turn_tokens = [role_token] + turn['text']
            context_tokens.extend(turn_tokens)
    
    return conv_samples


# 重新构建样本（保存文本）
print("构建新版本样本（保存原始文本）...")
train_rec_samples_v2 = build_recommendation_samples_v2(processed_train)
test_rec_samples_v2 = build_recommendation_samples_v2(processed_test)
train_conv_samples_v2 = build_conversation_samples_v2(processed_train)
test_conv_samples_v2 = build_conversation_samples_v2(processed_test)

print(f"\n✅ 新版本样本统计:")
print(f"  推荐训练: {len(train_rec_samples_v2)}")
print(f"  推荐测试: {len(test_rec_samples_v2)}")
print(f"  对话训练: {len(train_conv_samples_v2)}")
print(f"  对话测试: {len(test_conv_samples_v2)}")

print(f"\n示例推荐样本:")
print(f"  上下文: {train_rec_samples_v2[0]['context_text'][:100]}...")
print(f"  电影ID: {train_rec_samples_v2[0]['item']}")

print(f"\n示例对话样本:")
print(f"  上下文: {train_conv_samples_v2[0]['context_text'][:100]}...")
print(f"  回复: {train_conv_samples_v2[0]['response_text'][:100]}...")

In [ ]:
# 保存新格式数据（适配 LLaVA tokenizer）
import pickle

output_dir = 'data/redial_for_hypergraph_llava'
os.makedirs(output_dir, exist_ok=True)

# 保存推荐样本
with open(f'{output_dir}/train_rec_samples_text.pkl', 'wb') as f:
    pickle.dump(train_rec_samples_v2, f)

with open(f'{output_dir}/test_rec_samples_text.pkl', 'wb') as f:
    pickle.dump(test_rec_samples_v2, f)

# 保存对话样本
with open(f'{output_dir}/train_conv_samples_text.pkl', 'wb') as f:
    pickle.dump(train_conv_samples_v2, f)

with open(f'{output_dir}/test_conv_samples_text.pkl', 'wb') as f:
    pickle.dump(test_conv_samples_v2, f)

# 保存 side_data（与之前相同）
side_data = {
    'item2id': item2id,
    'id2item': {v: k for k, v in item2id.items()},
    'item_meta': {}  # 如果有电影元数据可以添加
}

with open(f'{output_dir}/side_data.pkl', 'wb') as f:
    pickle.dump(side_data, f)

print(f"✅ 数据已保存到 {output_dir}/")
print(f"  文件列表:")
print(f"    - train_rec_samples_text.pkl ({len(train_rec_samples_v2)} samples)")
print(f"    - test_rec_samples_text.pkl ({len(test_rec_samples_v2)} samples)")
print(f"    - train_conv_samples_text.pkl ({len(train_conv_samples_v2)} samples)")
print(f"    - test_conv_samples_text.pkl ({len(test_conv_samples_v2)} samples)")
print(f"    - side_data.pkl (item2id, id2item)")
print(f"\n📝 数据格式:")
print(f"  推荐样本: {{context_text: str, item: int, conv_id: int, turn_id: int}}")
print(f"  对话样本: {{context_text: str, response_text: str, conv_id: int, turn_id: int, role: str}}")
print(f"\n🚀 现在可以使用 HypergraphLlavaReDialDataset 加载这些数据了！")